# Notebook 3 — Feature engineering & labels

Per-artist weekly features from Last.fm events + Spotify; MSD audio joined on `artist_norm`; **binary label** = Billboard appearance with `week_date` in **(snapshot_week_end, snapshot_week_end + 28 days]** (next four calendar weeks).


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("."))
import hdfs_paths as hp

from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    sum as _sum,
    max as _max,
    lag,
    lower,
    regexp_replace,
    trim,
    to_date,
    year,
    weekofyear,
    broadcast,
    lit,
    avg,
)


def normalize_artist(c):
    return trim(lower(regexp_replace(c, "[^a-z0-9\\s]", "")))


spark = (
    SparkSession.builder.appName("MusicTrend_03_Features")
    .config("spark.sql.shuffle.partitions", hp.SHUFFLE_PARTITIONS)
    .getOrCreate()
)

events_df = spark.read.parquet(hp.PROCESSED_EVENTS).withColumn(
    "artist_norm", normalize_artist(col("artist_id"))
)
spotify_df = spark.read.parquet(hp.PROCESSED_SPOTIFY).withColumn(
    "artist_norm", normalize_artist(col("artist"))
)
billboard_df = spark.read.parquet(hp.PROCESSED_BILLBOARD).withColumn(
    "artist_norm", normalize_artist(col("performer"))
)
audio_df = spark.read.parquet(hp.PROCESSED_AUDIO).withColumn(
    "artist_norm", normalize_artist(col("artist_name"))
)

audio_by_artist = audio_df.groupBy("artist_norm").agg(
    avg("tempo").alias("tempo"),
    avg("energy").alias("energy"),
    avg("loudness").alias("loudness"),
    avg("danceability").alias("danceability"),
)


In [ ]:
# Artist-day aggregates + rolling windows (7d / 28d) and same-day plays as plays_1d proxy
from pyspark.sql.functions import sum as spark_sum

events_daily = (
    events_df.withColumn("event_date", to_date("event_timestamp"))
    .groupBy("artist_norm", "event_date")
    .agg(count("*").alias("daily_plays"))
)

W7 = (
    Window.partitionBy("artist_norm")
    .orderBy(col("event_date").cast("long"))
    .rangeBetween(-6 * 86400, 0)
)
W28 = (
    Window.partitionBy("artist_norm")
    .orderBy(col("event_date").cast("long"))
    .rangeBetween(-27 * 86400, 0)
)

features_daily = (
    events_daily.withColumn("plays_1d", col("daily_plays"))
    .withColumn("plays_7d", spark_sum("daily_plays").over(W7))
    .withColumn("plays_28d", spark_sum("daily_plays").over(W28))
    .withColumn(
        "growth_rate_7d",
        col("plays_7d") / (col("plays_28d") + lit(1)),
    )
)

# Weekly grain: take end-of-week snapshot = max(event_date) per ISO (week_year, week_number)
wk = (
    features_daily.withColumn("week_year", year("event_date"))
    .withColumn("week_number", weekofyear("event_date"))
)

weekly = (
    wk.groupBy("artist_norm", "week_year", "week_number")
    .agg(
        _max("plays_7d").alias("plays_7d"),
        _max("plays_28d").alias("plays_28d"),
        _max("growth_rate_7d").alias("growth_rate_7d"),
        _max("plays_1d").alias("plays_1d"),
        _max("event_date").alias("snapshot_end_date"),
    )
)


In [ ]:
# Spotify weekly: total streams, region spread, stream velocity
spotify_weekly = (
    spotify_df.withColumn("week_year", year("chart_date"))
    .withColumn("week_number", weekofyear("chart_date"))
    .groupBy("artist_norm", "week_year", "week_number")
    .agg(
        _sum("streams").alias("total_streams"),
        countDistinct("region").alias("region_spread"),
    )
)

w_art_week = Window.partitionBy("artist_norm").orderBy("week_year", "week_number")
spotify_vel = (
    spotify_weekly.withColumn("prev_streams", lag("total_streams", 1).over(w_art_week))
    .withColumn(
        "stream_velocity",
        (col("total_streams") - col("prev_streams")) / (col("prev_streams") + lit(1)),
    )
)

combined = (
    weekly.join(
        spotify_vel.select(
            "artist_norm",
            "week_year",
            "week_number",
            "total_streams",
            "region_spread",
            "stream_velocity",
        ),
        on=["artist_norm", "week_year", "week_number"],
        how="left",
    )
    .fillna(0, subset=["total_streams", "region_spread", "stream_velocity"])
)


In [ ]:
# Billboard label: any chart week in (snapshot_end, snapshot_end + 28d]
billboard_keys = billboard_df.select(
    "artist_norm", col("week_date").alias("billboard_week_date")
).distinct()

from pyspark.sql.functions import date_add

feat = combined.withColumn("label_window_end", date_add(col("snapshot_end_date"), 28))

hits = (
    feat.select(
        "artist_norm",
        "week_year",
        "week_number",
        "snapshot_end_date",
        "label_window_end",
    )
    .join(billboard_keys, on="artist_norm", how="left")
    .filter(col("billboard_week_date").isNotNull())
    .filter(col("billboard_week_date") > col("snapshot_end_date"))
    .filter(col("billboard_week_date") <= col("label_window_end"))
    .select("artist_norm", "week_year", "week_number")
    .distinct()
    .withColumn("charted", lit(1))
)

labeled = (
    combined.join(hits, on=["artist_norm", "week_year", "week_number"], how="left")
    .fillna(0, subset=["charted"])
    .join(broadcast(audio_by_artist), on="artist_norm", how="left")
)

labeled.select("charted").groupBy("charted").count().show()

labeled.write.mode("overwrite").parquet(hp.PROCESSED_FEATURES)
print("Wrote features to", hp.PROCESSED_FEATURES)


## Outputs confirmed

- `processed/features/` Parquet with labels and audio columns.
- Label distribution printed.
